> **ai use foreword;** the Lab is the part where the whole point is that you internalize why a number is trustworthy. If Claude builds your Sharpe calculation and you play with the output, you've learned the API, not the statistics. The test you already hold yourself to — "could I build this without AI?" — matters more here than anywhere

---

# CH 0. libraries & imports and

- numpy; sci computing
- pandas; data manupulation
- idk yet..


pull db to /notebooks

In [2]:
from fileinput import close

import yfinance
# pull db to /notebooks
!railway ssh cat /data/portfolio.db     > portfolio.db
!railway ssh cat /data/portfolio.db-wal > portfolio.db-wal

Using SSH key from file /Users/yazan/.ssh/id_ed25519.pub: yazan@crc-dot1x-nat-10-239-48-76.bu.edu
Using SSH key from file /Users/yazan/.ssh/id_ed25519.pub: yazan@crc-dot1x-nat-10-239-48-76.bu.edu


### turn to csv

In [3]:
import sqlite3, pandas as pd

con = sqlite3.connect("portfolio.db")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

df = pd.read_sql("SELECT * FROM trades", con)
df.to_csv("trades.csv", index=False)

In [4]:
df

,id,user_id,ticker,action,shares,trade_date,created_at,price
0,1,1,AAPL,buy,1.32,2024-08-20,2026-05-29T18:58:45.503411+00:00,NaN
1,2,1,AMZN,buy,1.12,2024-08-20,2026-05-29T18:58:45.503411+00:00,NaN
2,3,1,NVDA,buy,2.08,2024-08-20,2026-05-29T18:58:45.503411+00:00,NaN
3,4,1,VOO,buy,0.45,2024-08-20,2026-05-29T18:58:45.503411+00:00,NaN
4,5,1,GOOG,buy,1.00,2024-08-21,2026-05-29T18:58:45.503411+00:00,NaN
...,...,...,...,...,...,...,...,...
222,224,1,GOOG,sell,1.00,2026-06-05,2026-06-10T19:33:40.030242+00:00,NaN
223,225,1,LCID,buy,2.00,2026-06-10,2026-06-11T00:49:40.831357+00:00,NaN
224,226,1,SPCX,buy,1.00,2026-06-12,2026-06-12T16:25:39.958466+00:00,150.0
225,227,1,RKLB,sell,6.00,2026-06-12,2026-06-12T16:28:47.504491+00:00,NaN


# CH 1. Returns, volatility & Sharpe.

**Cover:**
- arithmetic vs log returns and why log returns add across time;
- annualization (√252) and why it's a convention with assumptions baked in;
- Sharpe as excess-return-per-unit-vol.

**Deliverable:** compute your portfolio's daily returns, annualized vol, and Sharpe by hand in numpy, then verify against a library. **Break it:** compute Sharpe on a short window, then a longer one — watch it swing wildly, and understand why Sharpe on <1yr of data is nearly meaningless. Also compute it on a trending stock vs a choppy one with the same total return.

---

## definitions

- **close price;** last traded price of the day
- **return;** fractional change in price over a period. normalizes prices in pp instead of $
- **price series;** a sequence of an asset's price, one observation per period.

- **arithmetic (simple) return;** `(p_today - p_yesterday) / p_yesterday`
    the logical return; issue is that it does not sum cleanly over time, a +10% then -10% is negative

- **log return;** `ln(P_today / P_yesterday)` todo why?

- **volatility;** sd of returns. how far price swings from average.
- **variance;** `sd²`
    - easier to work with

- **annualization;** scaling a daily number up to a yearly one, so figures are comparible.
    - scale by 252, roughly the amount of trading days in a year
    - volatility scales by ×√252 (the √ is because variance adds linearly over time but std is its square root)

- **sharpe ratio;** excess return per unit of volatility
    - answers how much return are you getting for the risk taken.
    - higher - better risk-adjusted performance
    - later distrusted

- **risk-free rate;** zero risk return (T-bills) todo any other? bonds? murabahah? etc
- **excess return;** return minus the risk-free rate. The "extra" you earned for taking risk.

**dependency chain:** 1. price series → 2. returns → 3. mean, volatility → 4. Sharpe

---

lets compute for `aapl` first

## 1. price series
lets do last 2 years of apple's closing price

In [52]:
import yfinance as yf
import pandas as pd
import numpy as np

# download AAPL, last 2 years, adjusted

aapl_price = yf.download('AAPL', period='2y', interval='1d', auto_adjust=True)

# grab just the Close column

aapl_price = aapl_price['Close'].squeeze()

aapl_price.head()





[*********************100%***********************]  1 of 1 completed


Date
2024-06-13    212.400085
2024-06-14    210.665146
2024-06-17    214.809235
2024-06-18    212.449661
2024-06-20    207.879242
Name: AAPL, dtype: float64

## 2. returns
we need the pp change per day, we'll do simple return

In [29]:
hist_A = aapl_price[1:].values[:] # prints arr of prices from 2nd element to last
hist_B = aapl_price[:-1].values[:] # prints first element to second last

# print(len(hist_A)) # 500
# print(len(hist_B)) # 500

# say we have 5 days of closing prices
# arr_A: 2 3 4 5
# arr_B: 1 2 3 4

# calc as (p_today - p_yesterday) / p_yesterday
aapl_simple_return = (hist_A - hist_B) / hist_B

print(type(aapl_price)) #<class 'pandas.core.frame.DataFrame'>
print(aapl_price.shape) # (501, 1)

# df's are more than 1d, theyre tabels. series (arrays) are 1d. use pandas .squeeze() to collapse single col df to a series

# appended .squeeze() to aapl_price

# <class 'pandas.core.series.Series'>
# (501,)

<class 'pandas.core.series.Series'>
(501,)


## 3. mean, volatility

lets get the mean and volatility (sd, var) of the aapl series!

In [50]:
aapl_mean = aapl_simple_return.mean()

print('%' + str(aapl_mean * 100)) # %0.07839517759774797
# meaning, on avg, aapl increased 0.078% every day in the selected period. net upward trend == increased. you may run this in the far future and see apple's stock actually decline over two years, maybe not. either way i hope you're happy


print(aapl_simple_return.max())   # biggest single day gain
print(aapl_simple_return.min())   # biggest single day loss

# volatility
print(aapl_simple_return.std())   # how far the typical day strays from that mean



%0.07839517759774797
0.15328862822026942
-0.09245612450870919
0.017569348918663387


## 4. Sharpe
**sharpe ratio;** excess return per unit of volatility. The industry convention is **annual**, so we annualize.

unit of vol is still in %, so we still use std, not var.
excess return = annualized return - annualized risk free return

we also need to annualize the std so its the same scale. sum of indep rv's var is additive, and to get std you sqrt
unit of volatility = (simple_return.std()) * sqrt(252))

In [58]:
risk_free = 0.03# pull smth from yfincance?


sharpe = ((simple_return.mean() * 252) - risk_free) / ((simple_return.std()) * np.sqrt(252))

print(sharpe) # 0.6007636548423583

0.6007636548423583


let's write a sharpe(ticker) function!!

good to know: The standard is the 13-week T-bill yield (ticker ^IRX on Yahoo Finance). It's what most practitioners use as the daily risk-free rate. we'll sub that in instead of the hard-coded 3%

In [70]:
risk_free = (yf.Ticker('^IRX').info['previousClose']) / 100 # 13-week T-bill yield

def sharpe(ticker):
    # takes a ticker. gets the last year's closing price series. squeezes it. computes mean, std. annualizes. computes sharpe as (ann_return - risk_free) / (return.std) * np.sqrt(252)
    price = yf.download(ticker, period='1y', interval='1d', auto_adjust=True)

    price = price['Close']

    price = price.squeeze()



    # return
    ret_A = price[1:].values[:]
    ret_B = price[:-1].values[:]

    price_return = (ret_A - ret_B) / ret_B

    return ((price_return.mean() * 252) - risk_free) / ((price_return.std()) * np.sqrt(252))

print(sharpe('AAPL'))

[*********************100%***********************]  1 of 1 completed

1.7234449297168215


- `< 0` — losing money risk-adjusted
- `0–1` — okay, you're being compensated but not great
- `1–2` — good
- `> 2` — excellent (or something's wrong with your data)



In [73]:
for ticker in ['aapl', 'spy', 'nvda', 'msft', 'lcid', 'rklb']:
    print(ticker + ': ' + str(sharpe(ticker)))

[*********************100%***********************]  1 of 1 completed


aapl: 1.7234450541276054


[*********************100%***********************]  1 of 1 completed


spy: 1.6460428192512713


[*********************100%***********************]  1 of 1 completed


...

msft: -0.7559770129055111


[*********************100%***********************]  1 of 1 completed


lcid: -1.4748025454650693


[*********************100%***********************]  1 of 1 completed

rklb: 1.9300751269914043


lets do sharpe by period

In [81]:
def sharpe_period(ticker, period):
    # takes a ticker. gets the last year's closing price series. squeezes it. computes mean, std. annualizes. computes sharpe as (ann_return - risk_free) / (return.std) * np.sqrt(252)
    price = yf.download(ticker, period=period, interval='1d', auto_adjust=True)

    price = price['Close']

    price = price.squeeze()

    # return
    ret_A = price[1:].values[:]
    ret_B = price[:-1].values[:]

    price_return = (ret_A - ret_B) / ret_B

    return ((price_return.mean() * 252) - risk_free) / ((price_return.std()) * np.sqrt(252))

print(sharpe_period('aapl', '10y'))
print(sharpe_period('aapl', '2y'))
print(sharpe_period('aapl', '1y'))
print(sharpe_period('aapl', '3mo'))
print(sharpe_period('aapl', '1mo'))
print(sharpe_period('aapl', '5d'))

[*********************100%***********************]  1 of 1 completed


0.9123351776335409


[*********************100%***********************]  1 of 1 completed


0.5784260937426209


[*********************100%***********************]  1 of 1 completed


...

2.6301601119711813


[*********************100%***********************]  1 of 1 completed


-1.435179800333995


[*********************100%***********************]  1 of 1 completed

-7.199733601092239


Sharpe on less than a year of data is nearly meaningless.

# CH 2. Data & reproducibility

This belongs before CAPM because every later unit consumes this data. Cover: what yfinance actually gives you (adjusted vs unadjusted close, how splits/dividends are handled), point-in-time correctness, and where survivorship sneaks in (your trades.csv only contains names you bought — you never see the ones you'd have bought and that died). Deliverable: a clean returns pipeline with a documented adjustment policy. Break it: pull an unadjusted series across a split (e.g., NVDA's 2024 10:1) and watch a fake 90% "loss" appear.

# CH 3. CAPM: alpha, beta, R²

Cover: beta as covariance(portfolio, market)/variance(market), alpha as the intercept, R² as how much of your variance the market explains. You already settled on beta-over-alpha for the dashboard because alpha is too noisy over 18 months — this unit is where you prove that to yourself rather than take it on faith. Deliverable: regress your daily returns on SPY, report β, α, R². Break it: compute alpha, then bootstrap-resample your returns and recompute it 1000 times — see the confidence interval swallow the point estimate. That's the noise, made visible.

# CH 4. Statistical significance & multiple testing

The hinge of the whole Lab. Cover: t-stats on a Sharpe, why testing 50 strategies and keeping the best one guarantees a false positive, the basic idea of deflating for trials (this is the López de Prado material — his "Deflated Sharpe" is exactly this). Deliverable: simulate 100 random strategies, find the "best," show its out-of-sample performance collapses. Break it: that is the break-it exercise. Do it once and you'll never trust a raw backtest again.

# CH 5. Backtesting pitfalls.

Now you have the tools to make this concrete instead of cautionary. Cover: look-ahead (using data you wouldn't have had), overfitting (you just lived it in #4), survivorship (you saw it in #2). Deliverable: a deliberately look-ahead-biased backtest of a simple rule, then the corrected version, side by side. Break it: introduce one line that peeks at tomorrow's close and watch returns go vertical.

# CH 6. The factor zoo

Cover: Fama-French size/value, momentum, quality — what each premium claims and the economic story behind it. Deliverable: regress your portfolio against FF factors (the data's free from Ken French's library) and see which exposures you actually carry — you'll likely find you're long momentum and growth without having chosen to be. Break it: run the regression on a random subset of dates vs another; watch the loadings move and ask which are real.

# CH 7. Position sizing

Cover: Kelly criterion and why full-Kelly is too aggressive for anyone sane, vol targeting as the practical alternative. Connect to what you already know about concentration widening the distribution without raising expected return. Deliverable: compute the vol-target weights your current portfolio implies vs what you actually hold. Break it: simulate full-Kelly on a strategy with mis-estimated edge and watch it blow up.

# CH 8. Options:

Cover: payoff diagrams first (you have this staged in Sprout already), then delta/theta/vega as sensitivities. This pairs with your ToS paper-trading plan. Deliverable: plot payoff diagrams for the basic structures, then a simple Black-Scholes greeks calculator. Break it: hold a position and watch theta decay it daily on paper